In [ ]:
# Install the project dependencies required by this notebook.
%pip install -q pandas numpy duckdb pyarrow scikit-learn xgboost mlflow matplotlib scipy joblib pyyaml

In [ ]:
# Mount Google Drive so Colab can access the private MIMIC-IV files and derived artifacts.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone or update the GitHub repository so the notebook can import the shared project code.
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mbakos95/aki-sentinel.git"
REPO_DIR = Path("/content/aki-sentinel")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

sys.path.insert(0, str(REPO_DIR))

In [ ]:
# Define the private MIMIC-IV and artifact locations used by the pipeline.
from pathlib import Path

MIMIC_ROOT = Path("/content/drive/MyDrive/MIMIC-IV")
HOSP_DIR = MIMIC_ROOT / "hosp"
ICU_DIR = MIMIC_ROOT / "icu"

PRIVATE_ROOT = Path("/content/drive/MyDrive/AKI-Sentinel-Private")
ARTIFACT_DIR = PRIVATE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import MLflow and the project modeling functions used for temporal splitting and model training.
import pandas as pd

from src.modeling import temporal_split, train_models

In [ ]:
# Load the engineered one-row-per-ICU-stay modeling dataset.
DATASET_PATH = ARTIFACT_DIR / "features" / "ml_dataset.parquet"
ml_dataset = pd.read_parquet(DATASET_PATH)
ml_dataset.shape

In [ ]:
# Split the cohort into approximate historical, validation, and later temporal groups using anchor-year groups.
train_df, val_df, test_df = temporal_split(ml_dataset)

print("Train:", train_df.shape, f"positive={train_df['target'].mean():.3%}")
print("Validation:", val_df.shape, f"positive={val_df['target'].mean():.3%}")
print("Test:", test_df.shape, f"positive={test_df['target'].mean():.3%}")

In [ ]:
# Train the interpretable logistic baseline and the XGBoost model while logging runs to private MLflow storage.
MODEL_DIR = ARTIFACT_DIR / "models"
MLFLOW_DIR = ARTIFACT_DIR / "mlruns"

models, feature_columns = train_models(
    train_df=train_df,
    model_dir=MODEL_DIR,
    mlflow_dir=MLFLOW_DIR,
)

list(models)

In [ ]:
# Save the temporal split rows and model feature list so later notebooks reproduce the exact experiment.
SPLIT_DIR = ARTIFACT_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(SPLIT_DIR / "train.parquet", index=False)
val_df.to_parquet(SPLIT_DIR / "validation.parquet", index=False)
test_df.to_parquet(SPLIT_DIR / "test.parquet", index=False)

pd.Series(feature_columns, name="feature").to_csv(
    SPLIT_DIR / "model_features.csv",
    index=False,
)